# 22.10 — Planning under uncertainty (MDP/POMDP planning)

When actions are uncertain, plan by backing up expected value; when state is hidden, plan over beliefs.

MDPs replace a single successor with transition probabilities, and POMDPs add uncertainty about the current state. The backup resembles minimax with expectation replacing the opponent's minimum.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt


@dataclass
class MDP:
    name: str
    states: List[str]
    actions: List[str]
    transitions: Dict[Tuple[str, str], List[Tuple[float, str]]]
    rewards: Dict[Tuple[str, str], float]
    gamma: float


def bellman_backup(values, state, action, transitions, rewards, gamma):
    expected = 0.0
    for probability, next_state in transitions[(state, action)]:
        expected += probability * values[next_state]
    return rewards.get((state, action), 0.0) + gamma * expected


def value_iteration(mdp, iterations=30):
    values = {state: 0.0 for state in mdp.states}
    backups = 0
    for _ in range(iterations):
        new_values = {}
        for state in mdp.states:
            qs = []
            for action in mdp.actions:
                if (state, action) in mdp.transitions:
                    qs.append(bellman_backup(values, state, action, mdp.transitions, mdp.rewards, mdp.gamma))
                    backups += 1
            new_values[state] = max(qs) if qs else 0.0
        values = new_values
    policy = {}
    for state in mdp.states:
        scored = []
        for action in mdp.actions:
            if (state, action) in mdp.transitions:
                scored.append((bellman_backup(values, state, action, mdp.transitions, mdp.rewards, mdp.gamma), action))
        if scored:
            policy[state] = max(scored)[1]
    return values, policy, backups


def belief_update(prior, likelihood):
    weights = []
    for p, like in zip(prior, likelihood):
        weights.append(p * like)
    normalizer = sum(weights)
    return [weight / normalizer for weight in weights], normalizer


def make_chain_mdp(name, n, slip, gamma=0.9):
    states = [f"S{i}" for i in range(n)]
    actions = ["right", "wait"]
    transitions = {}
    rewards = {}
    for index, state in enumerate(states):
        if index == n - 1:
            transitions[(state, "wait")] = [(1.0, state)]
            rewards[(state, "wait")] = 1.0
            continue
        transitions[(state, "right")] = [(1.0 - slip, states[index + 1]), (slip, state)]
        transitions[(state, "wait")] = [(1.0, state)]
        rewards[(state, "right")] = -0.02
        rewards[(state, "wait")] = -0.05
    return MDP(name, states, actions, transitions, rewards, gamma)


def mdp_ladder():
    return [
        make_chain_mdp("D1 two-state trace", 2, 0.2),
        make_chain_mdp("D2 deterministic-ish", 4, 0.05),
        make_chain_mdp("D3 stochastic chain", 7, 0.2),
        make_chain_mdp("D4 noisy grid surrogate", 10, 0.3),
        make_chain_mdp("D5 hidden-state scale", 14, 0.4),
    ]

## The concept, built once (D1)

The Bellman optimality backup is $V^*(s)=\max_a[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V^*(s')]$.

In [ ]:
values = {"low": 0.0, "high": 10.0, "s": 0.0}
transitions = {("s", "try"): [(0.2, "low"), (0.8, "high")]}
rewards = {("s", "try"): 1.0}
q_value = bellman_backup(values, "s", "try", transitions, rewards, 0.9)
expected_next = 0.2 * 0 + 0.8 * 10
print("expected", expected_next, "Q", q_value)
assert expected_next == 8.0
assert q_value == 8.2

Discount horizon and belief update are numeric sanity checks: delayed reward is $10\gamma^3$, and POMDP posteriors normalize unnormalized weights.

In [ ]:
discounts = [10 * gamma ** 3 for gamma in [0.1, 0.9, 0.99]]
posterior, normalizer = belief_update([0.6, 0.4], [0.2, 0.9])
rounded_discounts = [round(value, 3) for value in discounts]
rounded_posterior = [round(value, 2) for value in posterior]
print("discounts", rounded_discounts)
print("posterior", rounded_posterior, "normalizer", normalizer)
assert rounded_discounts == [0.01, 7.29, 9.703]
assert round(normalizer, 2) == 0.48
assert rounded_posterior == [0.25, 0.75]

## The dataset ladder

D1–D5 grow from a two-state trace to larger stochastic chains standing in for noisy navigation and hidden-state planning.

In [ ]:
ladder = mdp_ladder()
for mdp in ladder:
    print(mdp.name, "states", len(mdp.states), "actions", len(mdp.actions), "gamma", mdp.gamma, "sample", list(mdp.transitions.items())[:2])

## Run the SAME method across D1–D5

Metric: start-state value and Bellman backups expanded; policy quality is value relative to a converged reference.

In [ ]:
results = []
for mdp in ladder:
    values, policy, backups = value_iteration(mdp, iterations=30)
    reference, reference_policy, reference_backups = value_iteration(mdp, iterations=80)
    start = mdp.states[0]
    error = abs(reference[start] - values[start])
    results.append({"name": mdp.name, "value": values[start], "error": error, "backups": backups, "policy": policy.get(start)})
print("rung | V(start) | error | backups | policy")
for row in results:
    print(row["name"], round(row["value"], 3), round(row["error"], 4), row["backups"], row["policy"])

## Results visualization

Panels show value error and backup count. The curve summarizes uncertainty-planning quality versus state-space size.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for axis, row in zip(axes, results):
    axis.bar(["error", "backups"], [row["error"], row["backups"]], color=["tomato", "steelblue"])
    axis.set_title(row["name"].split()[0])
plt.show()
xs = list(range(1, len(results) + 1))
fig, axis = plt.subplots(figsize=(6, 3))
axis.plot(xs, [row["error"] for row in results], marker="o", label="value error")
axis.plot(xs, [row["backups"] for row in results], marker="s", label="backups")
axis.legend()
axis.set_xlabel("D rung")
plt.show()

## Pitfall on D5: deterministic best-successor backup

Replacing the expectation by a single best successor overstates reliability, and unnormalized beliefs are not beliefs. The fix is summing probabilities and normalizing observation weights.

In [ ]:
mdp = ladder[-1]
values = {state: index for index, state in enumerate(mdp.states)}
stochastic_q = bellman_backup(values, mdp.states[0], "right", mdp.transitions, mdp.rewards, mdp.gamma)
best_successor = max(values[next_state] for probability, next_state in mdp.transitions[(mdp.states[0], "right")])
deterministic_q = mdp.rewards[(mdp.states[0], "right")] + mdp.gamma * best_successor
posterior, normalizer = belief_update([0.6, 0.4], [0.2, 0.9])
print("stochastic", stochastic_q, "deterministic", deterministic_q)
print("posterior sum", sum(posterior), "normalizer", normalizer)
assert deterministic_q >= stochastic_q
assert round(sum(posterior), 6) == 1.0

## Evaluate it + Practice

- Metric: value error, policy quality, and backup count.
- No-skill baseline: deterministically take the best successor.
- Cheap sanity check: Q=8.2 and posterior is (0.25, 0.75).
- Ablation: replace expectation by max successor and D5 value inflates.
- Failure signals: transition probabilities do not sum to 1 or beliefs are unnormalized.

### Practice
Change slip probability and plot start-state value.

### Practice
Run fewer value-iteration passes and observe value error.

### Practice
Try a different observation likelihood and normalize the belief.